# Module 10: Cluster Computing — Running PyAutoLens on Slurm

## Learning to Autolens

---

**Purpose:** So far, every module has run on your laptop. That's fine for *teaching* — but when we tried to run the full Modules 04, 05, and 09 pipelines locally in parallel, the MacBook dropped to 0% idle with 47/48 GB of RAM in use and started thrashing swap. Real science runs on a cluster. This module shows how to take any tutorial notebook and turn it into a Slurm job on Harvard's **Cannon** cluster (FAS Research Computing). Worked examples are provided for **Modules 04, 05, and 09**, plus a **generic `fit_template.py`** (Section 8) for converting your own notebook.

**Prerequisites:**
- Modules 03–05 conceptually (we reuse their code, not their notebooks).
- A Cannon account. If you don't have one, apply via <https://www.rc.fas.harvard.edu/> — Harvard CfA affiliates qualify for shared partitions for free.
- Comfortable with `ssh`, `rsync`, and reading job logs.

**What you'll build:** a reusable pattern — `fit_*.py` + `submit_*.slurm` + `push/pull` rsync + `export_results.py` — that converts any notebook into a cluster job and ships the finished results back into the repo as lightweight, git-trackable PDFs and JSON.

---

## Table of Contents

1. [Why Move to a Cluster?](#1-why-cluster)
2. [The Three-Part Pattern](#2-pattern)
3. [Part A: Extract the Fit as a Standalone Script](#3-extract)
4. [Part B: Write the Slurm Submission Script](#4-slurm)
5. [Part C: Transfer Data with rsync](#5-rsync)
6. [Nautilus Checkpoint Resume](#6-resume)
7. [Worked Examples: Modules 04, 05, 09 on Cannon](#7-example)
8. [Converting Your Own Notebook](#8-convert)
9. [Results Viewer: Lightweight Artifacts for New Users](#9-viewer)
10. [Monitoring and Debugging](#10-monitoring)
11. [FASRC-Specific Notes](#11-fasrc)
12. [Exercises](#12-exercises)

---

## 1. Why Move to a Cluster? <a id="1-why-cluster"></a>

### The symptom

On a 2023 MacBook Pro (M2 Max, 12 cores), running Module 04 end-to-end takes:

| Stage | Parameters | Local wall-clock | What happened |
|-------|-----------|------------------|----------------|
| `chain/search_1` (SIS) | 3 | ~35 min | fine |
| `chain/search_2` (SIE + shear) | ~14 | **stalled > 15 h** | Nautilus bounds = 203 for hours |
| SLaM SOURCE LP | ~20 | ~2 h | fine |
| SLaM SOURCE PIX × 2 | — | ~3 h | fine |
| SLaM LIGHT LP | ~6 | ~1 h | fine |
| SLaM MASS TOTAL | ~7 | ~1.5 h | fine |

So the total is ~24 h of successful work plus one indefinitely-stuck search. On a cluster with 32+ cores, the same pipeline typically completes in 4–8 h and — critically — you can **walk away** from it.

### Three things clusters buy you

1. **More cores.** Nautilus (and dynesty) parallelize likelihood evaluations. `number_of_cores=16` on Cannon routinely gives 10–12× speedups over `number_of_cores=4` on a laptop.
2. **More RAM.** Pixelized source inversions build a $N_{\rm pix} \times N_{\rm pix}$ mapping matrix. 32 GB on a laptop is tight for 4k × 4k HST cutouts; Cannon nodes offer 256–512 GB routinely.
3. **Time independence.** You don't need your laptop open and unpaused. The job runs even if you close the lid or your wifi drops. Slurm resubmits automatically if a node dies.

### What you *don't* get for free

- **No GUI.** You cannot call `aplt.subplot_fit()` in the cluster script and expect a plot to pop up. All visualization happens back on your laptop after you `rsync` the results.
- **No `get_ipython()`.** The `%matplotlib inline` magic breaks. Remove all Jupyter magics from the extracted fit script.
- **Slow filesystem.** Most HPC scratch filesystems are NFS-backed. Writing thousands of small files (like each Nautilus checkpoint step) can dominate runtime. Use `$SCRATCH`, not `$HOME`, for Nautilus output.

---

## 2. The Three-Part Pattern <a id="2-pattern"></a>

Every cluster-ready PyAutoLens job has three pieces, plus a post-processor:

```
  my_notebook.ipynb           (stays on laptop — visualization/analysis only)
         │
         ▼   extract fit code
  fit_notebook.py             (runs on cluster — no magics, CLI-driven)
         │
         ▼   wrapped by
  submit_cannon.slurm         (SBATCH directives + module loads + MODULE env var)
         │
         ▼   data flow
  push_to_cannon.sh           (code + dataset + existing checkpoints → cluster)
  pull_from_cannon.sh         (Nautilus outputs → laptop for plotting)
         │
         ▼   after pull
  export_results.py           (Nautilus output → small PDFs/JSON in Modules/XX/results/)
                              → these ARE committed to git so new users see them
```

### Shipped scripts

This repo ships worked instances covering **Modules 04, 05, and 09** in `Modules/10_Cluster_Computing/scripts/`:

| File | Purpose |
|------|---------|
| `fit_module04.py` | Mod 04 — two-search chain + 5-stage SLaM pipeline on `simple`/`simple__no_lens_light` |
| `fit_module05.py` | Mod 05 — parametric + pixelized-source search pair with `SafeAnalysisImaging` |
| `fit_module09.py` | Mod 09 — full 5-stage MGE SLaM (SOURCE LP → SOURCE PIX × 2 → LIGHT LP → MASS TOTAL) |
| `submit_cannon.slurm` | Generic SBATCH script — dispatches on `MODULE` env var |
| `export_results.py` | Post-processor: per search, writes `fit_subplot.pdf`, `corner.pdf`, `info.txt`, `summary.json`, `samples.csv` |
| `push_to_cannon.sh` | rsync laptop → Cannon (includes preserved `checkpoint.hdf5` for auto-resume) |
| `pull_from_cannon.sh` | rsync Cannon → laptop (pulls `$SCRATCH/.../output/` into `cannon_output/`) |

### The one-line submit

Because `submit_cannon.slurm` keys off the `MODULE` env var, running any of the three modules is:

```bash
sbatch --export=ALL,MODULE=04 --job-name=mod04 submit_cannon.slurm
sbatch --export=ALL,MODULE=05 --job-name=mod05 submit_cannon.slurm
sbatch --export=ALL,MODULE=09 --job-name=mod09 --mem=64G --time=48:00:00 submit_cannon.slurm
```

Mod 09 gets more memory and time because it has 5 Nautilus searches including a PowerLaw MASS TOTAL stage.

---

## 3. Part A: Extract the Fit as a Standalone Script <a id="3-extract"></a>

### Start from `jupyter nbconvert`

The mechanical part:

```bash
jupyter nbconvert --to script \
    Modules/04_Search_Chaining_SLaM/04_search_chaining_slam.ipynb \
    --stdout > /tmp/mod04_raw.py
```

This gives you a `.py` file with all code cells concatenated. **Do not ship this directly** — it contains Jupyter magics, interactive plotting, and hard-coded relative paths. Use it as a starting point.

### Five edits you always need to make

1. **Strip magics and plotters.**
   ```python
   # remove these lines:
   get_ipython().run_line_magic('matplotlib', 'inline')
   dataset_plotter.subplot_dataset()
   fit_plotter.subplot_fit()
   ```
   The cluster has no display. Plots are a *post-hoc* laptop activity.

2. **Replace relative paths with CLI arguments.** The notebook uses `Path('../../autolens_workspace_original/dataset/imaging/simple')` which only resolves from the notebook's working directory. On the cluster, pass:
   ```python
   parser.add_argument('--dataset-root', type=Path, required=True)
   ```
   and let the Slurm script supply the absolute path.

3. **Wire `number_of_cores` to `$SLURM_CPUS_PER_TASK`.** PyAutoFit's `af.SettingsSearch` accepts `number_of_cores=`. On a cluster you want this to equal the cores Slurm allocated:
   ```python
   settings_search = af.SettingsSearch(
       path_prefix=...,
       unique_tag=...,
       number_of_cores=int(os.environ.get('SLURM_CPUS_PER_TASK', '1')),
   )
   ```

4. **Bump `n_live`.** The hung `search_2` on the laptop is the canonical failure mode — Nautilus with `n_live=75` can produce a numerically singular neural-bound covariance (the `numpy.linalg.cholesky: Matrix is not positive definite` crash we saw in Solution 04). On the cluster we can afford `n_live=100` to 150 for modest-dim searches and `n_live=200`+ for the full SLaM final stage, and we don't care about the extra runtime.

5. **`flush=True` on every `print`.** Slurm buffers stdout and flushes only when the buffer fills or the process exits. If the job crashes, unflushed output is *gone*. This one-liner has saved more debugging hours than any other trick:
   ```python
   print(f"[SLaM] SOURCE LP done in {dt:.1f} min", flush=True)
   ```

See `scripts/fit_module04.py` in this module for the full example.

In [ ]:
# Peek at the extracted fit script structure
!head -30 scripts/fit_module04.py

---

## 4. Part B: Write the Slurm Submission Script <a id="4-slurm"></a>

A Slurm script has two layers: **directives** (`#SBATCH` lines, parsed before the shell runs) and **body** (a regular bash script that Slurm invokes).

### Anatomy of `submit_cannon.slurm`

```bash
#!/bin/bash
#SBATCH --job-name=autolens_mod04      # shows in `squeue`
#SBATCH --partition=shared             # free-for-all Cannon partition
#SBATCH --account=siag_lab             # fast fairshare scheduling
#SBATCH --time=24:00:00                # HH:MM:SS wall limit
#SBATCH --nodes=1                      # single-node (don't need MPI)
#SBATCH --ntasks=1                     # one Python process
#SBATCH --cpus-per-task=16             # Nautilus uses these for likelihoods
#SBATCH --mem=32G                      # RAM. 32G is comfortable for `simple` dataset
#SBATCH --output=logs/mod04_%j.out     # %j = job ID
#SBATCH --error=logs/mod04_%j.err
#SBATCH --mail-type=BEGIN,END,FAIL     # get emails
```

### Choosing partition and time

| Job profile | Partition | Time | Reasoning |
|-------------|-----------|------|-----------|
| Quick test, ≤4 h, ≤8 cores | `test` or `shared` | `4:00:00` | Fast queue |
| Full SLaM pipeline, 8–24 h, 16+ cores | `shared` | `24:00:00` | Usually starts in <1 h |
| Real-data SLaM (HST/JWST), 24+ h | `shared` (long) or `serial_requeue` | `72:00:00` | `serial_requeue` is interruptible but much faster to dispatch |
| Many parallel cutouts (embarrassingly parallel) | `shared` with `--array=0-49` | per-job | One Slurm job per lens |

### Environment activation (inside the slurm body)

FASRC no longer ships a Python 3.12 Lmod module, so we bootstrap the conda env directly from Miniforge (installed once in `$HOME/miniforge3`). `submit_cannon.slurm` does this for you:

```bash
export CONDA_ENV="${CONDA_ENV:-autolens312}"
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate "$CONDA_ENV"
# sanity check the interpreter we actually ended up in:
python -c "import autolens as al; print('autolens', al.__version__)"
```

The `autolens312` env is Python 3.12 + autolens ≥ 2026.4.13 — created once in an interactive session (see Section 7, "One-time setup on Cannon"). Do **not** use the legacy `autolens` env (Python 3.11 + autolens 2026.2.26.4): it pre-dates `RectangularAdaptDensity`, `RectangularAdaptImage`, and `reg.Adapt`, all of which Modules 04 / 05 / 09 depend on.

In [ ]:
# Inspect the submission script
!cat scripts/submit_cannon.slurm

---

## 5. Part C: Transfer Data with rsync <a id="5-rsync"></a>

`rsync` is the right tool: incremental, resumable, and ignores files that haven't changed. Never use `scp -r` for this — it re-copies everything every time.

### What to push

```
Learning_to_Autolens/
  ├── Modules/                 ← code
  ├── Solutions/               ← not strictly needed on cluster, but cheap
  ├── slam_v2026.py            ← REQUIRED — Module 04 imports this
  ├── autolens_workspace_original/
  │     └── dataset/           ← REQUIRED — small FITS files
  └── Modules/04_.../output/   ← INCLUDE existing checkpoints for resume
```

### What to skip

- `.git/` — keep the cluster copy decoupled from your working tree.
- `.claude/`, `.ipynb_checkpoints/`, `__pycache__/` — noise.
- `autolens_workspace_latest/` — 400 MB+, only needed for Module 09. Push separately if/when you want to run it.
- `Output/` (top-level) — will be regenerated on the cluster into `$SCRATCH`.

### Pattern

The `scripts/push_to_cannon.sh` wrapper encodes the full include/exclude list. Always do a dry run first:

```bash
./scripts/push_to_cannon.sh            # dry run — shows what *would* transfer
./scripts/push_to_cannon.sh --go       # actual transfer
```

### Pulling results back

After the job finishes (email notification, or check via `sacct -j $JOB_ID`):

```bash
./scripts/pull_from_cannon.sh --go
```

This lands Nautilus outputs in `Modules/10_Cluster_Computing/cannon_output/` on the laptop. From there you can load them in a notebook:

```python
import autofit as af
search = af.Nautilus(
    path_prefix='Modules/10_Cluster_Computing/cannon_output/module_04/slam',
    name='source_lp[1]_light[lp]_mass[total]_source[lp]',
    unique_tag='simple',
)
result = search.result  # reconstructed from the pickled samples
```

---

## 6. Nautilus Checkpoint Resume <a id="6-resume"></a>

Nautilus writes a `checkpoint.hdf5` inside each search's `files/search_internal/` directory after every bound update. If the file exists when a search starts, Nautilus **automatically** resumes from it. There is no flag to enable or disable this — it just happens.

### What this means for you

1. **Slurm hit the time limit.** Resubmit the same `sbatch submit_cannon.slurm` — no changes needed. The finished searches skip instantly; the partial one resumes.
2. **A node crashed or you cancelled the job.** Same — resubmit.
3. **You ran locally until it stalled, then moved to the cluster.** This is what we did for Module 04:
   - Local run produced `Modules/04_.../output/output/module_04/chaining/simple__no_lens_light/search_2_sie_nolenslight/.../checkpoint.hdf5` (96 MB, 203 bounds).
   - `push_to_cannon.sh` includes this file in the rsync.
   - On the cluster, Nautilus sees the checkpoint and resumes — no duplicate work.

### Checkpoint structure (for reference)

Inside each search directory on the cluster after a run:

```
search_2_sie_nolenslight/
  └── <hash>/
      ├── image/                   ← best-fit image, residual, chi^2 maps
      ├── files/
      │   ├── samples.csv          ← posterior samples
      │   ├── search_internal/
      │   │   └── checkpoint.hdf5  ← Nautilus state (resume source)
      │   └── ...
      └── info/                    ← human-readable model.info
```

### When NOT to resume

If you change the model (add a parameter, re-bound a prior) *and* reuse the same `unique_tag`, the checkpoint's dimension will not match the new model and Nautilus will error on load. The fix: change `unique_tag` (or `name`) to a fresh string — Nautilus writes into a new hashed subdir, starting fresh.

---

## 7. Worked Examples: Modules 04, 05, 09 on Cannon <a id="7-example"></a>

End-to-end, from your laptop terminal (not the notebook).

### One-time setup on Cannon

```bash
ssh rcordovarosado@login.rc.fas.harvard.edu
salloc --account=siag_lab --partition=test --time=1:00:00 --cpus-per-task=4 --mem=8G

# Bootstrap Miniforge once (skip if $HOME/miniforge3 already exists)
curl -L -o Miniforge3.sh \
  "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-$(uname)-$(uname -m).sh"
bash Miniforge3.sh -b -p "$HOME/miniforge3"
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda init bash && exec bash           # picks up conda on next login

# Create the project env (Python 3.12 + autolens 2026.4.13+)
conda create -n autolens312 python=3.12 -y
conda activate autolens312
python -m pip install --upgrade pip
python -m pip install -r requirements.txt       # same requirements.txt as laptop
python -c "import autolens as al; print(al.__version__)"   # expect 2026.4.13.x
exit   # leave the interactive allocation
```

**Use `python -m pip`, not bare `pip`.** A PATH-shadowed `~/.local/bin/pip` on some Cannon accounts is tied to Python 3.10 and will silently install autolens into the wrong interpreter. The symptom is "`pip install` succeeds, `import autolens` fails."

### Every submission

```bash
# from the local repo root
cd ~/Documents/AGEL/Learning_to_Autolens

# 1. Push code + data + any existing checkpoint
./Modules/10_Cluster_Computing/scripts/push_to_cannon.sh --go

# 2. Submit the module you want (example: Mod 04)
ssh cannon "cd learning_to_autolens && \
    sbatch --export=ALL,MODULE=04 --job-name=mod04 \
           Modules/10_Cluster_Computing/scripts/submit_cannon.slurm"

# 3. Wait for email (or poll: `ssh cannon squeue -u rcordovarosado`)

# 4. Pull results + commit the lightweight artifacts
./Modules/10_Cluster_Computing/scripts/pull_from_cannon.sh --go
git add Modules/04_*/results/
git commit -m "feat: Module 04 cluster results"
```

The submit script automatically runs `export_results.py` at the end, so the artifacts are already built inside the pushed repo by the time you pull. You just commit and push the `results/` directory on the local side.

### Expected runtimes (Cannon `shared`, 16 cores / 32 GB)

| Module | Stages | Wall time | Note |
|--------|--------|-----------|------|
| **04** | 7 (2 chain + 5 SLaM) | ~5.5 h | search_2 resumes from local checkpoint (96 MB) |
| **05** | 2 (parametric + pixelized) | ~3 h | search_2 resumes from local checkpoint (1.9 MB) |
| **09** | 5 (full MGE SLaM) | ~8–12 h | search_1 resumes from local checkpoint (33 MB); PowerLaw MASS TOTAL is the longest stage |

Submit all three in parallel if you want — Cannon will run them on separate nodes.

### Preserved local checkpoints (2026-04-18)

The following checkpoints are preserved in the repo from the local runs we killed, and the rsync includes them:

| Module | Search | Size | Progress at kill |
|--------|--------|------|-------------------|
| 04 | `search_2_sie_nolenslight` | 96 MB | 203 bounds, stuck |
| 05 | `search2_pixelized_source` | 1.9 MB | early |
| 09 | `search1_mge_lens_sersic_source` | 33 MB | ~30% |
| Sol/04 | `search_1_sis_nolenslight` | 58 MB | crashed with LinAlgError |
| Sol/09 | `search1_mge_lens_sersic_source` | 30 MB | ~30% |

---

## 8. Converting Your Own Notebook <a id="8-convert"></a>

The three `fit_module{04,05,09}.py` scripts are worked examples, but the
generic workflow takes any lens-fit notebook — yours included — to the
cluster with minimal edits. `fit_template.py` captures every invariant
piece so you only have to fill in the parts that are genuinely yours.

### The four-step round-trip

```
  ┌─────────────────┐      push_to_cannon.sh --go
  │  Laptop         │  ────────────────────────────▶  ┌────────────────┐
  │                 │                                  │                │
  │  Notebook   ─┐  │                                  │  Cannon        │
  │  fit_moduleNN.py─┤                                 │                │
  │  submit (case NN)│                                 │  sbatch        │
  │                 │                                  │   submit.slurm │
  │                 │                                  │                │
  │                 │                                  │  Nautilus →    │
  │                 │                                  │    output/     │
  │                 │                                  │  export_results│
  │                 │                                  │   → results/   │
  │                 │  ◀────────────────────────────   │                │
  │  Modules/NN/    │      pull_from_cannon.sh --go   │                │
  │   results/      │                                  │                │
  │  Viewer cell    │                                  │                │
  │   renders       │                                  │                │
  └─────────────────┘                                  └────────────────┘
```

### Step 1 — Copy the template

```bash
cp Modules/10_Cluster_Computing/scripts/fit_template.py \
   Modules/10_Cluster_Computing/scripts/fit_moduleNN.py   # NN = your number
```

The template already has:

- `al.Imaging.from_fits` + circular mask loader
- A `_force_visualize(analysis, result, ...)` call after every fit —
  guarantees `image/fit.fits` is written so `export_results.py` can
  compute chi²-per-pixel and max|residual| (without this, on a resumed
  search, those fields come back `null` in `summary.json`)
- CLI flags (`--repo-root`, `--dataset-root`, `--output-root`,
  `--dataset-name`, `--n-live`) matching what `submit_cannon.slurm`
  passes
- `SLURM_CPUS_PER_TASK` → Nautilus `number_of_cores`
- `print(..., flush=True)` so Slurm logs show progress live

### Step 2 — Fill in `build(...)`

Open `fit_moduleNN.py` and replace the example SIE + Sersic search with
whatever your notebook does — a two-search chain, a SLaM pipeline, a
multi-component mass model. For SLaM, import the stage helpers from
`slam_v2026` (see `fit_module04.py` and `fit_module09.py` for full
examples). For a simple single search, you can leave the structure alone
and just change the `model = af.Collection(...)` block.

### Step 3 — Add a `MODULE=NN` case to `submit_cannon.slurm`

Open `Modules/10_Cluster_Computing/scripts/submit_cannon.slurm` and
add a line to the dataset-root dispatch:

```bash
case "${MODULE}" in
    04|05) DATASET_ROOT="${REPO_ROOT}/autolens_workspace_original/dataset/imaging" ;;
    09)    DATASET_ROOT="${REPO_ROOT}/autolens_workspace_latest/dataset/imaging"  ;;
    NN)    DATASET_ROOT="${REPO_ROOT}/path/to/your/dataset"                        ;;
    *)     echo "unknown MODULE=${MODULE}"; exit 2 ;;
esac
```

Nothing else to edit in the slurm script — it already dispatches to
`fit_module${MODULE}.py` automatically. `push_to_cannon.sh` uses an
include-everything / exclude-list pattern, so any new `Modules/NN_*`
directory is automatically picked up; `pull_from_cannon.sh` uses a
`Modules/*/results/**` include pattern, so your module's
`results/` directory is automatically pulled back. No edits needed to
either rsync script.

### Step 4 — Add a results-viewer cell to your notebook

Copy the pattern from Modules 04 / 05 / 09 — a markdown cell titled
**"Viewing pre-computed results from the Cannon cluster"** followed
by a code cell that loads
`Modules/NN_<name>/results/<search_name>/` and displays
`fit_subplot.png`, `corner.pdf`, and `summary.json`. The artifacts
are git-tracked, so anyone who clones the repo sees your finished
fit without needing a Cannon account.

### The commands, in order

```bash
# Laptop — one-time for a new module
cp scripts/fit_template.py scripts/fit_moduleNN.py
# (edit fit_moduleNN.py's build() + add a MODULE=NN case to submit_cannon.slurm)

# Every submission
bash Modules/10_Cluster_Computing/scripts/push_to_cannon.sh --go
ssh cannon "cd learning_to_autolens && \
    sbatch --export=ALL,MODULE=NN --job-name=modNN \
           Modules/10_Cluster_Computing/scripts/submit_cannon.slurm"

# Wait for the email ("BEGIN" then "END").

# Pull + commit
bash Modules/10_Cluster_Computing/scripts/pull_from_cannon.sh --go
git add Modules/NN_<name>/results/
git commit -m "feat: Module NN cluster results"
```

### Quality check on the way back

Every time you pull a finished run, open
`Modules/NN/results/<stage>/fit_subplot.png` and `summary.json` and
verify both **before** advancing. The `autolens-fit-diagnostics` skill
(auto-triggers on any `results/` path in this repo) applies the numeric
thresholds in `references/thresholds.md` and the visual pattern catalog
in `references/residual-patterns.md`. A cluster run that converged to
the wrong local minimum (e.g., Pattern 1 — coherent arc in the residual
map) is still a failed fit no matter how long it ran. Don't trust the
scalar metrics alone — always open `fit_subplot.png` too.

### Safety guards already in place

Three invariants that `fit_template.py` inherits automatically:

1. **`PYAUTOFIT_TEST_MODE` detection** — if this env var is set,
   PyAutoFit skips sampling and returns a random prior draw. The slurm
   script refuses to submit; `slam_v2026.py` raises at import; every
   module notebook's imports cell raises on load; `check_install.py`
   flags it. You cannot accidentally run a "fit" that does nothing.
2. **Nautilus checkpoint auto-resume** — identical `path_prefix` +
   `name` + `unique_tag` + model hash picks up `checkpoint.hdf5` and
   continues. Just resubmit the same `sbatch` line.
3. **`_force_visualize(...)` after every `search.fit()`** — guarantees
   `image/fit.fits` exists for the export step, even on a resumed
   search.

---

## 9. Results Viewer: Lightweight Artifacts for New Users <a id="9-viewer"></a>

### The problem

A full Nautilus output directory for one search is 100–500 MB (posterior samples, per-iteration checkpoint, image grids, pickled state). For Module 09 with five searches it easily exceeds 2 GB. You cannot — and should not — commit that to git.

But a new user cloning the repo needs to *see the finished results*. They shouldn't have to set up a Cannon account and wait 12 h just to view a residual map.

### The solution

`scripts/export_results.py` walks the Nautilus output tree and, for each completed search, writes five small artifacts into `Modules/XX_.../results/<search_name>/`:

| Artifact | Typical size | What it is |
|----------|-------------|------------|
| `fit_subplot.pdf` | ~200 KB | Standard `FitImagingPlotter.subplot_fit()` — data, model, residuals, normalized residuals |
| `corner.pdf` | ~500 KB | Posterior corner plot from `NestPlotter.corner_cornerpy()` |
| `info.txt` | ~5 KB | Human-readable `result.info` — model structure, max-LL parameter values |
| `summary.json` | ~1 KB | `{max_log_likelihood, log_evidence, n_live, n_samples, ...}` |
| `samples.csv` | ~1–5 MB | Full Nautilus samples table (optional — keeps corner plots reproducible) |

Total per module: **~5–20 MB**. Easily git-trackable. Committed directly to `Modules/XX/results/`.

### Usage

The Slurm script already runs this automatically at the end of every job — no manual step. But if you want to re-export (e.g., after pulling a re-run):

```bash
python Modules/10_Cluster_Computing/scripts/export_results.py \
    --repo-root   $PWD \
    --module      04 \
    --output-root Modules/10_Cluster_Computing/cannon_output/module_04
```

Or for a single search:

```bash
python export_results.py \
    --search-dir Modules/10_Cluster_Computing/cannon_output/module_04/slam/source_lp[1]/<hash> \
    --dest       Modules/04_Search_Chaining_SLaM/results/source_lp_1
```

### Loading results in a module notebook

Each module notebook can surface the cluster-run results without any PyAutoFit machinery — just by loading the artifacts directly. A typical loader cell:

```python
from pathlib import Path
from IPython.display import display, Image, Markdown
import json

results_dir = Path("results")   # relative to the notebook

for search in sorted(results_dir.iterdir()):
    if not search.is_dir(): continue
    display(Markdown(f"### {search.name}"))
    with open(search / "summary.json") as f:
        s = json.load(f)
    display(Markdown(f"- max log likelihood: **{s['max_log_likelihood']:.2f}**"))
    display(Markdown(f"- log evidence: {s.get('log_evidence')}"))
    display(Image(str(search / "fit_subplot.pdf")))   # PDF may need a renderer
    display(Image(str(search / "corner.pdf")))
```

If PDF rendering in the notebook is flaky, convert to PNG during export or include a `jupyter nbconvert --to html` step in the submit script — but 99% of the time the PDFs render fine.

### Why not just commit the `cannon_output/` directory?

Three reasons:
1. **Size.** One module's raw output is a full-sized git-history pollutant.
2. **Reproducibility.** The lightweight artifacts encode the *result*, not the per-iteration state. If anyone wants to re-sample the posterior, they re-run on the cluster.
3. **Privacy of scratch.** `cannon_output/` paths reveal your Cannon directory structure; the artifacts don't.

---

## 10. Monitoring and Debugging <a id="10-monitoring"></a>

### Commands cheat sheet

```bash
squeue -u $USER                  # all my jobs
squeue -j $JOB_ID                # one job (state, node, time used)
sacct -j $JOB_ID                 # post-mortem: exit code, max mem, CPU time
seff $JOB_ID                     # pretty efficiency report
scontrol show job $JOB_ID        # full dump

scancel $JOB_ID                  # kill it
scancel -u $USER                 # kill ALL my jobs (nuclear option)

tail -f logs/mod04_$JOB_ID.out   # live stdout (if `flush=True` is set!)
tail -f logs/mod04_$JOB_ID.err   # live stderr
```

### Common failure modes

| Symptom | Cause | Fix |
|---------|-------|-----|
| Job dies immediately with `ModuleNotFoundError: autolens` | conda env not activated | Add `source activate autolens` before the `srun` line |
| `TIMEOUT` in sacct | hit wall limit | Resubmit; Nautilus resumes automatically |
| `OOM` / `OUT_OF_MEMORY` | mem too low | Raise `--mem=` — try 64G, 128G |
| `LinAlgError: Matrix not positive definite` | `n_live` too low | Bump `n_live` by 25–50 |
| No output in `.out`, job `RUNNING` for hours | `print` without `flush=True` | Fix the script; `scancel`; resubmit |
| `FileNotFoundError: data.fits` | path arg wrong | Check `$DATASET_ROOT` in Slurm script |
| `ModuleNotFoundError: slam_v2026` | `--repo-root` wrong | Check that `slam_v2026.py` is in the pushed tree |

---

## 11. FASRC-Specific Notes <a id="11-fasrc"></a>

### Filesystems

| Path | Size | Persistence | Use for |
|------|------|-------------|---------|
| `$HOME` | 100 GB | permanent, backed up | code, small configs, conda envs |
| `$SCRATCH` (`/n/holyscratch01/...`) | 50 TB | **purged after 90 days** | Nautilus outputs, checkpoints, intermediate files |
| `/n/holylabs/LABS/<pi>/Lab` | varies | permanent, not backed up | final published results |

Rule of thumb: code in `$HOME`, scratch-heavy I/O to `$SCRATCH`, archived results to `Lab/`.

**Do not** write Nautilus outputs to `$HOME` — the quota is too small and the I/O is slow.

### Partitions (as of 2026)

| Partition | Max time | Cores/node | Queue speed | Notes |
|-----------|----------|------------|-------------|-------|
| `test` | 8 h | up to 8 | seconds | Debugging |
| `shared` | 7 d | up to 48 | minutes | Workhorse |
| `serial_requeue` | 7 d | up to 48 | seconds | **Can be killed and requeued** — fine for checkpointed work |
| `gpu` | 7 d | 32 + A100 | minutes | Not currently used by PyAutoLens |

`serial_requeue` is the hidden gem for Nautilus jobs: it dispatches fast because it volunteers to be killed, and since we have checkpoint resume, killing-and-requeuing is basically free. Use it once you trust your checkpoint setup.

### SSH config

Add this to `~/.ssh/config` locally to avoid typing your username and host every time:

```
Host cannon
    HostName login.rc.fas.harvard.edu
    User rcordovarosado
    ServerAliveInterval 60
    ControlMaster auto
    ControlPath ~/.ssh/cm-%r@%h:%p
    ControlPersist 10m
```

Then: `ssh cannon`, `rsync -a ... cannon:~/...`, etc. The `ControlMaster` block reuses one SSH connection across rsync calls — dramatic speedup.

### Two-factor

Cannon requires Duo 2FA on every login unless you have an SSH key + Kerberos. For frequent syncing, generate a key and register it per FASRC docs; otherwise budget 20 seconds per `push_to_cannon.sh` for the Duo push.

---

## 12. Exercises <a id="12-exercises"></a>

### Exercise 1: Module 03 on the cluster

Module 03 ran fine locally, but it's a good practice target. Write `fit_module03.py` by `nbconvert`-ing Module 03, following the five edits in Section 3. Extend `submit_cannon.slurm` to accept `MODULE=03`. Submit, pull results, and confirm the corner plot (from `results/.../corner.pdf`) matches the one you got locally.

### Exercise 2: Array job for multiple lenses

Cannon supports job arrays. Modify the Slurm script so that a single `sbatch` launches N parallel jobs, one per target, via `#SBATCH --array=0-9`. Inside the script, read `$SLURM_ARRAY_TASK_ID` and use it to index into a list of dataset names. This is the pattern you'll want when we get to Module 07 (real data: AGEL lens sample).

### Exercise 3: Measure the speedup

Time the SOURCE LP stage on your laptop and on Cannon at `--cpus-per-task=4, 8, 16, 32`. Plot wall time vs. cores. Nautilus scales sublinearly past ~16 cores because the neural-bound update is serial — identify roughly where your diminishing returns kick in.

### Exercise 4: Kill-and-resume drill

Submit a short run. After ~10 min, `scancel` it. Verify the `checkpoint.hdf5` is non-empty. Resubmit the same script unchanged. Confirm from the new job's log that Nautilus reports a non-zero starting bound count — that's proof it resumed.

### Exercise 5: Extend `export_results.py`

Add a new artifact type: for pixelized-source searches only, render the source-plane reconstruction as a PNG and save it to `results/<search>/source_plane.png`. Hint: `fit.inversion.mapped_reconstructed_image` gives you the source-plane image; use `aplt.InversionPlotter`. This is useful for Module 05 and Module 09 stages 2–3.

---

## Summary

| Piece | File | Role |
|-------|------|------|
| Fit code | `scripts/fit_module{04,05,09}.py` | The notebook, minus magics and plotters, plus CLI + `flush=True` |
| Slurm | `scripts/submit_cannon.slurm` | Generic; dispatches on `MODULE` env var |
| Push | `scripts/push_to_cannon.sh` | rsync laptop → cluster (includes checkpoints for resume) |
| Pull | `scripts/pull_from_cannon.sh` | rsync cluster → laptop |
| Post-process | `scripts/export_results.py` | Extract PDFs + JSON into `Modules/XX/results/` (git-trackable) |
| Resume | (implicit, via `checkpoint.hdf5`) | Nautilus does this automatically |
| View results | Load `results/<search>/{fit_subplot.pdf, corner.pdf, summary.json}` from any module notebook | New users see finished fits without a cluster account |

**Next module:** Module 07 (real FITS data). Everything here applies directly — an AGEL lens with a 4k × 4k HST cutout is *exactly* when the cluster stops being optional.

---

*Learning to Autolens — Module 10*
*Rodrigo Córdova Rosado, Harvard CfA*
*Built with Claude Code*